# EDA очищенных CDR

Запускайте после `python -m src.preprocess`. Каждая ячейка вызывает функцию из `src/eda.py` — посмотрите её код, чтобы понять, как устроен расчёт.

In [ ]:
import sys
sys.path.insert(0, '..')  # чтобы работал import src

import pandas as pd
from src import eda

pd.set_option('display.float_format', '{:,.4g}'.format)
calls = eda.load_calls('../data/processed/calls.parquet')
calls.info(memory_usage='deep')

## 1. Общая картина

In [ ]:
eda.overview(calls)

## 2. Смысл типов записей

Есть ли у 03 номер C? У кого есть IMSI? Записан ли один звонок дважды (02 у A и 01 у B)?

In [ ]:
eda.type_semantics(calls)

In [ ]:
eda.mirrored_records(calls)

## 3. Когда звонят

In [ ]:
eda.calls_by_day_hour(calls)

## 4. Длительность

Нулевая длительность ≠ неотвеченный: статуса ответа в данных нет.

In [ ]:
eda.duration_summary(calls)

In [ ]:
eda.duration_by_category(calls)

## 5. Операторы

In [ ]:
eda.operator_matrix(calls)

In [ ]:
eda.unknown_codes(calls)

## 6. Активность номеров

In [ ]:
eda.activity_distribution(calls)

In [ ]:
top = eda.top_callers(calls, n=20)
top

In [ ]:
# устойчив ли паттерн по дням 1–5 октября
eda.daily_stability(calls, top.index)

## 7. SIM: номера на одной IMSI

In [ ]:
eda.sim_check(calls)

## 8. Графики

In [ ]:
%matplotlib inline
from IPython.display import Image, display
for p in eda.save_figures(calls, '../reports/eda'):
    display(Image(str(p)))

## 9. Признаки по номерам

После `python -m src.features`.

In [ ]:
feats = pd.read_parquet('../data/processed/features.parquet')
desc = pd.read_csv('../data/processed/features_description.csv', index_col=0)
callers = feats[feats['n_out'] > 0]
callers.describe(percentiles=[.5, .9, .99, .999]).T

In [ ]:
# самые «массовые»: много разных получателей, короткие звонки, нет перезвонов
callers.query('uniq_b >= 100').sort_values('peak_1h', ascending=False).head(30)